# Binary classification — data preparation

Dataset columns:
- **id** — object identifier (several rows per id)
- **gb** — binary target (0/1)
- **cat_*** — categorical features (integer codes)
- **num_*** — numerical features (many missing values)

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("/Users/zaharguzij/privat_test/train_df.csv")
TARGET_COL = "gb"  # binary target; rename here if your file uses "db"
ID_COL = "id"

In [5]:
def detect_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Return (categorical, numerical) column names by prefix."""
    cat_cols = [c for c in df.columns if c.startswith("cat_")]
    num_cols = [c for c in df.columns if c.startswith("num_")]
    return cat_cols, num_cols


def load_raw_data(path: Path | str = DATA_PATH, sep: str = "\t") -> pd.DataFrame:
    df = pd.read_csv(path, sep=sep)
    if TARGET_COL not in df.columns:
        raise KeyError(f"Target column '{TARGET_COL}' not found. Columns: {list(df.columns[-5:])}")
    return df


def prepare_binary_classification_data(
    df: pd.DataFrame,
    *,
    target_col: str = TARGET_COL,
    id_col: str = ID_COL,
    max_onehot_cardinality: int = 20,
    scale_numeric: bool = True,
    drop_id_from_features: bool = True,
) -> dict:
    """
    Build feature matrix X, target y, and a fitted sklearn preprocessor.

    Returns dict with keys: X, y, ids, feature_names, preprocessor, meta.
    """
    cat_cols, all_num_cols = detect_feature_columns(df)

    y = df[target_col].astype(int).values
    ids = df[id_col].values

    # Drop numerical columns that are entirely missing (cannot be imputed)
    num_cols = [c for c in all_num_cols if df[c].notna().any()]
    dropped_num = [c for c in all_num_cols if c not in num_cols]

    low_card_cat = [c for c in cat_cols if df[c].nunique() <= max_onehot_cardinality]
    high_card_cat = [c for c in cat_cols if c not in low_card_cat]

    transformers = []

    if num_cols:
        num_steps: list[tuple[str, object]] = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            num_steps.append(("scaler", StandardScaler()))
        transformers.append(("num", Pipeline(num_steps), num_cols))

    if low_card_cat:
        transformers.append(
            (
                "cat_low",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                low_card_cat,
            )
        )

    # High-cardinality categoricals: keep integer codes (good for tree models)
    if high_card_cat:
        transformers.append(("cat_high", "passthrough", high_card_cat))

    preprocessor = ColumnTransformer(transformers, remainder="drop")
    X = preprocessor.fit_transform(df[cat_cols + num_cols])

    # Readable feature names after one-hot encoding
    feature_names: list[str] = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat_low":
            ohe: OneHotEncoder = trans  # type: ignore[assignment]
            feature_names.extend(ohe.get_feature_names_out(cols).tolist())
        elif name == "cat_high":
            feature_names.extend(cols)

    meta = {
        "n_rows": len(df),
        "n_features": X.shape[1],
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "low_card_cat": low_card_cat,
        "high_card_cat": high_card_cat,
        "dropped_all_nan_num": dropped_num,
        "class_balance": pd.Series(y).value_counts(normalize=True).to_dict(),
        "target_col": target_col,
        "id_col": id_col,
        "drop_id_from_features": drop_id_from_features,
    }

    return {
        "X": X,
        "y": y,
        "ids": ids,
        "feature_names": feature_names,
        "preprocessor": preprocessor,
        "meta": meta,
    }


def train_val_split(
    prepared: dict,
    *,
    test_size: float = 0.2,
    random_state: int = 42,
    stratify: bool = True,
    split_by_id: bool = True,
) -> dict:
    """Split prepared data; use split_by_id=True to avoid id leakage across folds."""
    X, y, ids = prepared["X"], prepared["y"], prepared["ids"]

    if split_by_id:
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, val_idx = next(splitter.split(X, y, groups=ids))
    else:
        train_idx, val_idx = train_test_split(
            np.arange(len(y)),
            test_size=test_size,
            random_state=random_state,
            stratify=y if stratify else None,
        )

    return {
        "X_train": X[train_idx],
        "X_val": X[val_idx],
        "y_train": y[train_idx],
        "y_val": y[val_idx],
        "ids_train": ids[train_idx],
        "ids_val": ids[val_idx],
        "train_idx": train_idx,
        "val_idx": val_idx,
    }

In [6]:
df = load_raw_data()
cat_cols, num_cols = detect_feature_columns(df)

print(f"rows: {len(df):,}, unique ids: {df[ID_COL].nunique():,}")
print(f"features: {len(cat_cols)} categorical, {len(num_cols)} numerical")
print(f"target {TARGET_COL}:\n{df[TARGET_COL].value_counts()}")

prepared = prepare_binary_classification_data(df)
split = train_val_split(prepared, split_by_id=True)

print(f"\nX shape: {prepared['X'].shape}")
print(f"train: {split['X_train'].shape}, val: {split['X_val'].shape}")
print(f"class balance (train): {pd.Series(split['y_train']).value_counts(normalize=True).round(4).to_dict()}")
print(f"high-card cat kept as integers: {prepared['meta']['high_card_cat'][:5]} ...")

rows: 26,824, unique ids: 5,243
features: 135 categorical, 416 numerical
target gb:
gb
0    26231
1      593
Name: count, dtype: int64

X shape: (26824, 868)
train: (21473, 868), val: (5351, 868)
class balance (train): {0: 0.9769, 1: 0.0231}
high-card cat kept as integers: ['cat_19', 'cat_21', 'cat_38', 'cat_60', 'cat_77'] ...


## Logistic regression

Linear baseline on the prepared features. `class_weight="balanced"` compensates for the rare positive class (~2%).

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)


def train_logistic_regression(
    split: dict,
    *,
    C: float = 1.0,
    max_iter: int = 2000,
    random_state: int = 42,
) -> dict:
    """Fit logistic regression and return model + validation metrics."""
    model = LogisticRegression(
        C=C,
        class_weight="balanced",
        max_iter=max_iter,
        random_state=random_state,
    )
    model.fit(split["X_train"], split["y_train"])

    y_prob = model.predict_proba(split["X_val"])[:, 1]
    y_pred = model.predict(split["X_val"])

    metrics = {
        "roc_auc": roc_auc_score(split["y_val"], y_prob),
        "avg_precision": average_precision_score(split["y_val"], y_prob),
        "confusion_matrix": confusion_matrix(split["y_val"], y_pred),
        "classification_report": classification_report(split["y_val"], y_pred, digits=4),
    }
    return {"model": model, "y_prob": y_prob, "y_pred": y_pred, "metrics": metrics}

In [ ]:
lr_result = train_logistic_regression(split)
lr_model = lr_result["model"]
metrics = lr_result["metrics"]

print("Validation metrics (logistic regression)")
print(f"ROC-AUC:          {metrics['roc_auc']:.4f}")
print(f"Average precision:{metrics['avg_precision']:.4f}")
print("\nConfusion matrix [ [TN, FP], [FN, TP] ]:")
print(metrics["confusion_matrix"])
print("\nClassification report:")
print(metrics["classification_report"])
print()
# Top coefficients (largest absolute weights)
coef = pd.Series(lr_model.coef_.ravel(), index=prepared["feature_names"])
print("\nTop 10 positive coefficients:")
print(coef.nlargest(10).round(4))
print("\nTop 10 negative coefficients:")
print(coef.nsmallest(10).round(4))

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_linea

Validation metrics (logistic regression)
ROC-AUC:          0.7828
Average precision:0.1190

Confusion matrix [ [TN, FP], [FN, TP] ]:
[[4710  543]
 [  48   50]]

Classification report:
              precision    recall  f1-score   support

           0     0.9899    0.8966    0.9410      5253
           1     0.0843    0.5102    0.1447        98

    accuracy                         0.8896      5351
   macro avg     0.5371    0.7034    0.5428      5351
weighted avg     0.9733    0.8896    0.9264      5351


Top 10 positive coefficients:
cat_73_3     3.1027
cat_128_3    2.3856
cat_78_3     2.0395
num_9        1.8418
num_177      1.8061
num_414      1.7821
cat_81_3     1.7435
cat_126_3    1.7353
cat_65_1     1.7231
num_93       1.5764
dtype: float64

Top 10 negative coefficients:
cat_73_13   -3.0905
num_341     -2.2547
num_52      -2.1100
num_320     -2.0204
cat_73_11   -1.8683
num_332     -1.8100
num_220     -1.7783
cat_126_1   -1.6706
num_41      -1.6499
cat_128_1   -1.6454
dtype: float

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Pytho

## Tree models — XGBoost, Random Forest, CatBoost

Uses **raw** `cat_*` / `num_*` features (median imputation on numerics only; no scaling / one-hot).  
Each model is trained with several hyperparameter presets; results are ranked by validation ROC-AUC.

In [9]:
import time
from itertools import product

from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier


def prepare_tree_model_data(
    df: pd.DataFrame,
    *,
    target_col: str = TARGET_COL,
    id_col: str = ID_COL,
) -> dict:
    """Features for tree boosting: categoricals as integers, numerics median-imputed."""
    cat_cols, all_num_cols = detect_feature_columns(df)
    num_cols = [c for c in all_num_cols if df[c].notna().any()]

    X_df = df[cat_cols + num_cols].copy()
    for col in cat_cols:
        X_df[col] = X_df[col].astype(int)
    for col in num_cols:
        X_df[col] = X_df[col].fillna(X_df[col].median())

    y = df[target_col].astype(int).values
    ids = df[id_col].values
    cat_feature_indices = list(range(len(cat_cols)))

    return {
        "X": X_df.values,
        "X_df": X_df,
        "y": y,
        "ids": ids,
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "cat_feature_indices": cat_feature_indices,
        "feature_names": cat_cols + num_cols,
    }


def _eval_predictions(y_true, y_prob, y_pred, fit_sec: float, config_name: str, model_name: str) -> dict:
    return {
        "model": model_name,
        "config": config_name,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "avg_precision": average_precision_score(y_true, y_prob),
        "fit_sec": round(fit_sec, 1),
        "tn": int(((y_true == 0) & (y_pred == 0)).sum()),
        "fp": int(((y_true == 0) & (y_pred == 1)).sum()),
        "fn": int(((y_true == 1) & (y_pred == 0)).sum()),
        "tp": int(((y_true == 1) & (y_pred == 1)).sum()),
    }


def run_xgboost_configs(tree_split: dict, configs: list[dict], *, scale_pos_weight: float | None = None) -> pd.DataFrame:
    if scale_pos_weight is None:
        n_neg = (tree_split["y_train"] == 0).sum()
        n_pos = (tree_split["y_train"] == 1).sum()
        scale_pos_weight = n_neg / max(n_pos, 1)

    rows = []
    for cfg in configs:
        name = cfg.pop("name")
        params = dict(cfg)
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=1,
            **params,
        )
        t0 = time.perf_counter()
        model.fit(tree_split["X_train"], tree_split["y_train"])
        fit_sec = time.perf_counter() - t0
        y_prob = model.predict_proba(tree_split["X_val"])[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        rows.append(_eval_predictions(tree_split["y_val"], y_prob, y_pred, fit_sec, name, "xgboost"))
        cfg["name"] = name
    return pd.DataFrame(rows)


def run_random_forest_configs(tree_split: dict, configs: list[dict]) -> pd.DataFrame:
    rows = []
    for cfg in configs:
        name = cfg.pop("name")
        model = RandomForestClassifier(random_state=42, n_jobs=1, **cfg)
        t0 = time.perf_counter()
        model.fit(tree_split["X_train"], tree_split["y_train"])
        fit_sec = time.perf_counter() - t0
        y_prob = model.predict_proba(tree_split["X_val"])[:, 1]
        y_pred = model.predict(tree_split["X_val"])
        rows.append(_eval_predictions(tree_split["y_val"], y_prob, y_pred, fit_sec, name, "random_forest"))
        cfg["name"] = name
    return pd.DataFrame(rows)


def run_catboost_configs(
    tree_prepared: dict,
    tree_split: dict,
    configs: list[dict],
) -> pd.DataFrame:
    train_pool = Pool(
        tree_split["X_df_train"],
        tree_split["y_train"],
        cat_features=tree_prepared["cat_cols"],
    )
    val_pool = Pool(
        tree_split["X_df_val"],
        tree_split["y_val"],
        cat_features=tree_prepared["cat_cols"],
    )

    rows = []
    for cfg in configs:
        name = cfg.pop("name")
        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=0,
            allow_writing_files=False,
            **cfg,
        )
        t0 = time.perf_counter()
        model.fit(train_pool, eval_set=val_pool, use_best_model=True)
        fit_sec = time.perf_counter() - t0
        y_prob = model.predict_proba(val_pool)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        rows.append(_eval_predictions(tree_split["y_val"], y_prob, y_pred, fit_sec, name, "catboost"))
        cfg["name"] = name
    return pd.DataFrame(rows)

In [10]:
# --- Hyperparameter presets (expand grids below for more combinations) ---

XGBOOST_CONFIGS = [
    {"name": "xgb_shallow_fast", "n_estimators": 300, "max_depth": 4, "learning_rate": 0.1, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5, "reg_alpha": 0.0, "reg_lambda": 1.0},
    {"name": "xgb_shallow_slow", "n_estimators": 800, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.9, "min_child_weight": 3, "reg_alpha": 0.1, "reg_lambda": 1.0},
    {"name": "xgb_medium", "n_estimators": 500, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 3, "reg_alpha": 0.0, "reg_lambda": 2.0},
    {"name": "xgb_medium_reg", "n_estimators": 600, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.7, "colsample_bytree": 0.7, "min_child_weight": 10, "reg_alpha": 1.0, "reg_lambda": 5.0},
    {"name": "xgb_deep", "n_estimators": 400, "max_depth": 8, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1, "reg_alpha": 0.0, "reg_lambda": 1.0},
    {"name": "xgb_deep_reg", "n_estimators": 500, "max_depth": 8, "learning_rate": 0.03, "subsample": 0.75, "colsample_bytree": 0.75, "min_child_weight": 5, "reg_alpha": 2.0, "reg_lambda": 8.0},
    {"name": "xgb_aggressive", "n_estimators": 1000, "max_depth": 10, "learning_rate": 0.02, "subsample": 0.9, "colsample_bytree": 0.9, "min_child_weight": 1, "reg_alpha": 0.0, "reg_lambda": 0.5},
    {"name": "xgb_conservative", "n_estimators": 1200, "max_depth": 5, "learning_rate": 0.01, "subsample": 0.6, "colsample_bytree": 0.6, "min_child_weight": 20, "reg_alpha": 5.0, "reg_lambda": 10.0},
]

RANDOM_FOREST_CONFIGS = [
    {"name": "rf_default", "n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt", "class_weight": "balanced"},
    {"name": "rf_shallow", "n_estimators": 400, "max_depth": 8, "min_samples_split": 10, "min_samples_leaf": 5, "max_features": "sqrt", "class_weight": "balanced"},
    {"name": "rf_deep", "n_estimators": 500, "max_depth": 20, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "sqrt", "class_weight": "balanced_subsample"},
    {"name": "rf_wide", "n_estimators": 600, "max_depth": 12, "min_samples_split": 5, "min_samples_leaf": 2, "max_features": 0.5, "class_weight": "balanced"},
    {"name": "rf_strict_leaf", "n_estimators": 500, "max_depth": 10, "min_samples_split": 20, "min_samples_leaf": 10, "max_features": "log2", "class_weight": "balanced"},
    {"name": "rf_many_trees", "n_estimators": 1000, "max_depth": 14, "min_samples_split": 4, "min_samples_leaf": 2, "max_features": 0.3, "class_weight": "balanced_subsample"},
    {"name": "rf_log_features", "n_estimators": 700, "max_depth": None, "min_samples_split": 5, "min_samples_leaf": 3, "max_features": "log2", "class_weight": "balanced"},
    {"name": "rf_extra_trees_style", "n_estimators": 800, "max_depth": 16, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": 1.0, "class_weight": "balanced", "bootstrap": True, "max_samples": 0.8},
]

CATBOOST_CONFIGS = [
    {"name": "cb_default", "iterations": 500, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 50},
    {"name": "cb_shallow", "iterations": 600, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 5.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 50},
    {"name": "cb_deep", "iterations": 500, "depth": 8, "learning_rate": 0.05, "l2_leaf_reg": 3.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 50},
    {"name": "cb_slow", "iterations": 1200, "depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 6.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 80},
    {"name": "cb_fast", "iterations": 300, "depth": 6, "learning_rate": 0.1, "l2_leaf_reg": 2.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 30},
    {"name": "cb_high_reg", "iterations": 700, "depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 15.0, "min_data_in_leaf": 50, "auto_class_weights": "Balanced", "early_stopping_rounds": 60},
    {"name": "cb_bayesian", "iterations": 600, "depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3.0, "bagging_temperature": 0.8, "random_strength": 1.5, "auto_class_weights": "Balanced", "early_stopping_rounds": 50},
    {"name": "cb_aggressive", "iterations": 1000, "depth": 10, "learning_rate": 0.03, "l2_leaf_reg": 1.0, "auto_class_weights": "Balanced", "early_stopping_rounds": 80},
]

# Optional: auto-expand XGBoost grid (depth x learning_rate x n_estimators)
_EXPAND_XGB = False
if _EXPAND_XGB:
    extra = []
    for depth, lr, n_est, subsample in product([4, 6, 8], [0.03, 0.05, 0.1], [400, 700], [0.7, 0.85]):
        extra.append({
            "name": f"xgb_grid_d{depth}_lr{lr}_n{n_est}",
            "max_depth": depth,
            "learning_rate": lr,
            "n_estimators": n_est,
            "subsample": subsample,
            "colsample_bytree": subsample,
            "min_child_weight": 5,
            "reg_alpha": 0.5,
            "reg_lambda": 3.0,
        })
    XGBOOST_CONFIGS = XGBOOST_CONFIGS + extra

print(f"configs to train: xgb={len(XGBOOST_CONFIGS)}, rf={len(RANDOM_FOREST_CONFIGS)}, catboost={len(CATBOOST_CONFIGS)}")

configs to train: xgb=8, rf=8, catboost=8


In [11]:
# Prepare tree features and reuse the same train/val indices as logistic regression
tree_prepared = prepare_tree_model_data(df)
train_idx, val_idx = split["train_idx"], split["val_idx"]
tree_split = {
    "X_train": tree_prepared["X"][train_idx],
    "X_val": tree_prepared["X"][val_idx],
    "X_df_train": tree_prepared["X_df"].iloc[train_idx],
    "X_df_val": tree_prepared["X_df"].iloc[val_idx],
    "y_train": tree_prepared["y"][train_idx],
    "y_val": tree_prepared["y"][val_idx],
}

print(f"tree features: {tree_prepared['X'].shape[1]} ({len(tree_prepared['cat_cols'])} cat + {len(tree_prepared['num_cols'])} num)")

results_xgb = run_xgboost_configs(tree_split, [c.copy() for c in XGBOOST_CONFIGS])
results_rf = run_random_forest_configs(tree_split, [c.copy() for c in RANDOM_FOREST_CONFIGS])
results_cb = run_catboost_configs(tree_prepared, tree_split, [c.copy() for c in CATBOOST_CONFIGS])

tree_results = (
    pd.concat([results_xgb, results_rf, results_cb], ignore_index=True)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Leaderboard (validation, sorted by ROC-AUC) ===")
tree_results

best = tree_results.iloc[0]
print(f"\nBest: {best['model']} / {best['config']} — ROC-AUC={best['roc_auc']:.4f}, AP={best['avg_precision']:.4f}")

tree features: 539 (135 cat + 404 num)


KeyboardInterrupt: 

In [ ]:
# Retrain best tree model and show full classification report
best_model_name = best["model"]
best_config_name = best["config"]

if best_model_name == "xgboost":
    cfg = next(c for c in XGBOOST_CONFIGS if c["name"] == best_config_name)
    n_neg = (tree_split["y_train"] == 0).sum()
    n_pos = (tree_split["y_train"] == 1).sum()
    best_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        scale_pos_weight=n_neg / max(n_pos, 1),
        random_state=42,
        **{k: v for k, v in cfg.items() if k != "name"},
    )
    best_model.fit(tree_split["X_train"], tree_split["y_train"])
    y_prob_best = best_model.predict_proba(tree_split["X_val"])[:, 1]

elif best_model_name == "random_forest":
    cfg = next(c for c in RANDOM_FOREST_CONFIGS if c["name"] == best_config_name)
    best_model = RandomForestClassifier(random_state=42, **{k: v for k, v in cfg.items() if k != "name"})
    best_model.fit(tree_split["X_train"], tree_split["y_train"])
    y_prob_best = best_model.predict_proba(tree_split["X_val"])[:, 1]

else:
    cfg = next(c for c in CATBOOST_CONFIGS if c["name"] == best_config_name)
    train_pool = Pool(tree_split["X_df_train"], tree_split["y_train"], cat_features=tree_prepared["cat_cols"])
    val_pool = Pool(tree_split["X_df_val"], tree_split["y_val"], cat_features=tree_prepared["cat_cols"])
    best_model = CatBoostClassifier(random_seed=42, verbose=0, allow_writing_files=False, **{k: v for k, v in cfg.items() if k != "name"})
    best_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    y_prob_best = best_model.predict_proba(val_pool)[:, 1]

y_pred_best = (y_prob_best >= 0.5).astype(int)
print(classification_report(tree_split["y_val"], y_pred_best, digits=4))
print("Confusion matrix:\n", confusion_matrix(tree_split["y_val"], y_pred_best))

## Neural network — best performance (final task)

Tabular **embedding network**: each `cat_*` column gets its own embedding; numerics are scaled + missing flags.  
Trains several architectures, ensembles the best one (multi-seed), then **blends** with the top tree model for the overall champion score.

In [ ]:
NN_CONFIGS = [
    {"name": "nn_small", "hidden": (256, 128), "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"name": "nn_medium", "hidden": (512, 256, 128), "dropout": 0.30, "lr": 1e-3, "weight_decay": 1e-4},
    {"name": "nn_large", "hidden": (768, 384, 192), "dropout": 0.35, "lr": 8e-4, "weight_decay": 2e-4},
    {"name": "nn_wide", "hidden": (1024, 512), "dropout": 0.40, "lr": 5e-4, "weight_decay": 3e-4},
    {"name": "nn_deep", "hidden": (512, 512, 256, 128), "dropout": 0.30, "lr": 7e-4, "weight_decay": 2e-4},
]

nn_data = prepare_nn_data(df)
print(f"NN input: {len(nn_data['cat_cols'])} cat embeddings + {nn_data['X_num'].shape[1]} numeric (incl. missing flags)")
print(f"Device: {DEVICE}")

nn_rows = []
nn_models = {}
for cfg in NN_CONFIGS:
    name = cfg["name"]
    print(f"Training {name}...")
    result = train_tabular_nn(
        nn_data,
        split,
        hidden=cfg["hidden"],
        dropout=cfg["dropout"],
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"],
        seed=42,
    )
    nn_models[name] = result
    nn_rows.append({
        "model": "neural_net",
        "config": name,
        "roc_auc": result["roc_auc"],
        "avg_precision": result["avg_precision"],
        "fit_sec": result["fit_sec"],
    })

nn_results = pd.DataFrame(nn_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
print("\n=== Neural network configs ===")
nn_results

In [ ]:
# Multi-seed ensemble for the best NN architecture
best_nn_name = nn_results.iloc[0]["config"]
best_cfg = next(c for c in NN_CONFIGS if c["name"] == best_nn_name)
ENSEMBLE_SEEDS = [42, 7, 2024, 1337]

print(f"Ensembling '{best_nn_name}' with seeds {ENSEMBLE_SEEDS}...")
seed_probs = []
for seed in ENSEMBLE_SEEDS:
    r = train_tabular_nn(nn_data, split, seed=seed, **{k: v for k, v in best_cfg.items() if k != "name"})
    seed_probs.append(r["y_prob"])
    print(f"  seed {seed}: ROC-AUC={r['roc_auc']:.4f}")

nn_ensemble_prob = np.mean(seed_probs, axis=0)
y_val = nn_data["y"][split["val_idx"]]
nn_ens_auc = roc_auc_score(y_val, nn_ensemble_prob)
nn_ens_ap = average_precision_score(y_val, nn_ensemble_prob)
print(f"\nNN ensemble ROC-AUC: {nn_ens_auc:.4f} | AP: {nn_ens_ap:.4f}")

In [ ]:
# Blend NN ensemble with best tree model (weight tuned on validation)
try:
    tree_prob = y_prob_best  # from best-tree cell
except NameError:
    raise RuntimeError("Run the tree-model cells first (train leaderboard + retrain best tree).")

best_blend_auc, best_w = 0.0, 0.5
for w in np.linspace(0.0, 1.0, 41):
    blend = w * nn_ensemble_prob + (1 - w) * tree_prob
    auc = roc_auc_score(y_val, blend)
    if auc > best_blend_auc:
        best_blend_auc, best_w = auc, w

champion_prob = best_w * nn_ensemble_prob + (1 - best_w) * tree_prob
champion_thr, champion_f1 = find_best_threshold(y_val, champion_prob)
champion_pred = (champion_prob >= champion_thr).astype(int)

champion_metrics = {
    "model": "champion_blend",
    "config": f"nn_ensemble({best_nn_name})*{best_w:.2f} + {best_model_name}({best_config_name})*{1-best_w:.2f}",
    "roc_auc": roc_auc_score(y_val, champion_prob),
    "avg_precision": average_precision_score(y_val, champion_prob),
    "threshold": champion_thr,
    "f1_at_thr": champion_f1,
}

print("=== CHAMPION (NN + best tree blend) ===")
print(f"Blend weight (NN): {best_w:.2f}")
print(f"ROC-AUC:          {champion_metrics['roc_auc']:.4f}")
print(f"Average precision:{champion_metrics['avg_precision']:.4f}")
print(f"Best threshold:   {champion_thr:.3f} (F1={champion_f1:.4f})")
print("\nConfusion matrix:\n", confusion_matrix(y_val, champion_pred))
print("\nClassification report:")
print(classification_report(y_val, champion_pred, digits=4))

In [ ]:
# Grand leaderboard — all approaches
all_results = []

# Logistic regression
if "lr_result" in dir():
    all_results.append({
        "model": "logistic_regression",
        "config": "balanced",
        "roc_auc": lr_result["metrics"]["roc_auc"],
        "avg_precision": lr_result["metrics"]["avg_precision"],
    })

# Tree models
if "tree_results" in dir():
    all_results.append(tree_results.assign(rank_type="tree")[["model", "config", "roc_auc", "avg_precision"]])

# Neural nets
all_results.append(nn_results[["model", "config", "roc_auc", "avg_precision"]])
all_results.append(pd.DataFrame([{
    "model": "neural_net",
    "config": f"ensemble_{best_nn_name}",
    "roc_auc": nn_ens_auc,
    "avg_precision": nn_ens_ap,
}]))
all_results.append(pd.DataFrame([{
    "model": champion_metrics["model"],
    "config": champion_metrics["config"],
    "roc_auc": champion_metrics["roc_auc"],
    "avg_precision": champion_metrics["avg_precision"],
}]))

final_leaderboard = (
    pd.concat(all_results, ignore_index=True)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

print("=" * 60)
print("FINAL LEADERBOARD — all models (validation ROC-AUC)")
print("=" * 60)
final_leaderboard